# CryoSat-2 subsetting, filtering, and block median

Run the sections in order: export station-centered raw subsets, apply the per-window standard-deviation filter, then block-median the filtered points.


# Export station-centered CryoSat-2 subsets

This section creates raw spatial and temporal subsets for the paper stations and CryoSat-2 products.


In [ ]:
from __future__ import annotations

"""Export unfiltered station-centered satellite subsets.

This isolates the SPATIOTEMPORAL_ONLY export step of the
processing script. It keeps the same temporal windows, circular radius,
source files, and CSV metadata. No point deduplication, quality flag
filtering, iterative sigma clipping, or block-median processing is done.

Existing subset CSVs are left unchanged. Remove or relocate an old output
file before rerunning if you actually intend to regenerate that window.
"""

from pathlib import Path
from typing import Dict, List, Optional, Tuple, Union, Any, NamedTuple
import re
import threading
import time
import concurrent.futures as cf

import numpy as np
import pandas as pd
from pyproj import Transformer
import vaex

# ----------------------------------------------------------------------
# Paths and datasets
# ----------------------------------------------------------------------
DO_MISWIS = True
DO_POCA = True
DO_SWATH = True

# Set OUT_ROOT to the folder containing the station/dataset subset tree.
OUT_ROOT = Path("/Volumes/cornice/marquardt/data/interpWindow")
SPATIOTEMPORAL_SUBDIR = "SPATIOTEMPORAL_ONLY"
EXPORT_SPATIOTEMPORAL_SUBSETS = True

# Matches original October 3 window selection
TIME_WINDOW_DAYS = 90
STEP_DAYS = 30
POST_DATE_CUTOFF = pd.Timestamp("2011-01-01")
RADIUS_COL = "r02_m"

# This is kriging_windows_master.csv exported by generate_cryosat2_kriging_windows_master.py.
ICEFALL_KRIGING_WINDOWS_MASTER_CSV = Path(
    "/Volumes/icefall/marquardt/results/bias_results/kriging_windows_master.csv"
)
# This is rema_variogram_params_station_midpoint.csv from the REMA R script.
ICEFALL_VARIOGRAM_CSV = Path(
    "/Volumes/icefall/marquardt/results/bias_results/rema_variogram_params_station_midpoint.csv"
)
# Point these to the monthly MISWIS, POCA, and SWATH parquet input folders.
MISWIS_MONTHLY_DIRS = [
    Path("/Volumes/cornice/marquardt/data/CS2/SIR_SIN_L2/monthly"),
    Path("/Volumes/cornice/marquardt/data/CS2/miswis"),
]
MISWIS_PREFIX = "cs2_SINL2"

POCA_MONTHLY_DIR = Path(
    "/Volumes/cornice/marquardt/data/CS2/cs2_tempo_LI/TEMPO_POCA_LI_monthly_parquet"
)
POCA_PREFIX = "POCA_LI"

SWATH_MONTHLY_DIR = Path(
    "/Volumes/cornice/marquardt/data/CS2/cs2_tempo_LI/TEMPO_SWATH_POINT_monthly_parquet"
)
SWATH_PREFIX = "SWATH_POINT"
SWATH_TIME_ORIGIN = pd.Timestamp("1970-01-01")


# Logging and parallelism
PARALLEL_MAX_WORKERS = 1
WINDOW_MAX_WORKERS = 1


# Same coordinate and cache setup as the original code
_ll2xy_3031 = Transformer.from_crs("EPSG:4326", "EPSG:3031", always_xy=True).transform
_ll2xy_3413 = Transformer.from_crs("EPSG:4326", "EPSG:3413", always_xy=True).transform
_xy2ll_3031 = Transformer.from_crs("EPSG:3031", "EPSG:4326", always_xy=True).transform
_xy2ll_3413 = Transformer.from_crs("EPSG:3413", "EPSG:4326", always_xy=True).transform
_PRINT_LOCK = threading.Lock()
_local_cache = threading.local()
_MISWIS_CACHE_MAX = 24
_POCA_CACHE_MAX = 32
_SWATH_CACHE_MAX = 64

# -----------------------------
# Coordinate and CSV helpers
# -----------------------------
def xy_epsg_for_dem_name(dem_name: Any) -> int:
    s = str(dem_name).strip().upper()
    if "ARCTIC" in s:
        return 3413
    return 3031

def project_lonlat_to_xy(lon: float, lat: float, epsg: int) -> Tuple[float, float]:
    if epsg == 3413:
        x, y = _ll2xy_3413(float(lon), float(lat))
    else:
        x, y = _ll2xy_3031(float(lon), float(lat))
    return float(x), float(y)

def project_xy_to_lonlat(x: float, y: float, epsg: int) -> Tuple[float, float]:
    """
    Convert the projected x_mid/y_mid from the kriging-window master back to
    """
    if epsg == 3413:
        lon, lat = _xy2ll_3413(float(x), float(y))
    else:
        lon, lat = _xy2ll_3031(float(x), float(y))
    return float(lon), float(lat)

def regions_for_dem_name(dem_name: Any) -> Tuple[List[str], List[str]]:
    s = str(dem_name).strip().upper()
    if "ARCTIC" in s:
        return ["GREENL"], ["GREENLAND"]
    return ["ANTARC"], ["ANTARCTIC"]

def _p(msg: str) -> None:
    with _PRINT_LOCK:
        print(msg, flush=True)

def safe_makedirs(p: Path) -> Path:
    p.mkdir(parents=True, exist_ok=True)
    return p

def parse_time_series(s: pd.Series, origin: pd.Timestamp = pd.Timestamp("2000-01-01")) -> pd.Series:
    if pd.api.types.is_datetime64_any_dtype(s):
        return s
    if pd.api.types.is_numeric_dtype(s):
        return pd.to_datetime(s, unit="s", origin=origin, errors="coerce")
    return pd.to_datetime(s, errors="coerce")

def circle_subset_with_cap(df: pd.DataFrame, x_mid: float, y_mid: float, radius_m: float, cap_points: int):
    if df is None or df.empty:
        return df.iloc[0:0].copy(), float("nan"), False
    if "x" not in df.columns or "y" not in df.columns:
        return df.iloc[0:0].copy(), float("nan"), False

    x = pd.to_numeric(df["x"], errors="coerce").to_numpy(float)
    y = pd.to_numeric(df["y"], errors="coerce").to_numpy(float)
    ok = np.isfinite(x) & np.isfinite(y)
    if not ok.any():
        return df.iloc[0:0].copy(), float("nan"), False

    dx = x[ok] - float(x_mid)
    dy = y[ok] - float(y_mid)
    d2 = dx*dx + dy*dy

    r2 = float(radius_m) * float(radius_m)
    in_circle = d2 <= r2
    if not in_circle.any():
        return df.iloc[0:0].copy(), float("nan"), False

    ok_pos = np.flatnonzero(ok)
    in_pos = ok_pos[np.flatnonzero(in_circle)]
    sub = df.iloc[in_pos].copy().reset_index(drop=True)

    if len(sub) <= cap_points:
        return sub, float(radius_m), False

    dist = np.sqrt(d2[in_circle])
    k = int(cap_points)
    take = np.argpartition(dist, kth=k-1)[:k]
    take = take[np.argsort(dist[take])]
    sub_cap = df.iloc[in_pos[take]].copy().reset_index(drop=True)
    radius_used = float(np.max(dist[take]))
    return sub_cap, radius_used, True

def export_to_csv(df: pd.DataFrame, out_csv: Path, attrs: Dict[str, Any]) -> None:
    if df is None or df.empty:
        return
    safe_makedirs(out_csv.parent)
    with open(out_csv, "w", encoding="utf-8") as f:
        f.write("# =================================================\n")
        f.write("# Station window export - CSV format\n")
        f.write("# =================================================\n")
        for k, v in sorted(attrs.items()):
            if isinstance(v, pd.Timestamp):
                v = v.isoformat()
            f.write(f"# {k}: {v}\n")
        f.write("# =================================================\n")
        df.to_csv(f, index=False)
    _p(f"  [WROTE] {out_csv} rows={len(df)}")  # NEW: definitive write confirmation

def spatiotemporal_out_csv(out_dir: Path, win_id: str) -> Path:
    return out_dir / SPATIOTEMPORAL_SUBDIR / f"{win_id}_SPATIOTEMPORAL_ONLY.csv"

def export_spatiotemporal_subset(
    *,
    ds: str,
    stU: str,
    win_id: str,
    out_dir: Path,
    df: pd.DataFrame,
    attrs: Dict[str, Any],
) -> Tuple[bool, str]:
    """
    Export the dataset immediately after the requested temporal + circular
    spatial subset, before deduplication, quality filtering, sigma clipping,
    or point caps. Unlike final exports, this does not require
    MIN_POINTS_PER_WINDOW.
    """
    if not EXPORT_SPATIOTEMPORAL_SUBSETS:
        return False, "disabled"

    out_csv = spatiotemporal_out_csv(out_dir, win_id)

    if out_csv.exists():
        return False, "skip_exists"

    if df is None or df.empty:
        return False, "skip_empty"

    raw_attrs = dict(attrs)
    raw_attrs.update(
        export_stage="spatial_temporal_subset_before_filtering",
        deduplication_applied="NO",
        quality_filtering_applied="NO",
        sigma_clipping_applied="NO",
        point_cap_applied="NO",
        n_points=int(len(df)),
    )
    export_to_csv(df.copy(), out_csv, raw_attrs)
    return True, "exported"

def norm_station(s: str) -> str:
    s = str(s).strip().upper()
    s = re.sub(r"[^A-Z0-9]", "", s)

    # Preserve the actual station IDs without zero-padding.
    if re.fullmatch(r"GLS[123]", s):
        return s

    if re.fullmatch(r"SOW[234]", s):
        return s

    return s

def load_kriging_windows_master(path: Path) -> pd.DataFrame:
    """
    Load the all-station kriging-window master.

    Required columns:
      station, mid, x_mid, y_mid

    Optional columns used when present:
      w0, w1, win_id/window_id, time_window_days, step_days, variogram,
      r02_m, points_csv/cs2_points_csv

    The exact projected x_mid/y_mid values in this file are used directly as
    each window's spatial center. No lat_mean/lon_mean interpolation-center CSV
    is used.
    """
    if not path.exists():
        raise FileNotFoundError(f"Kriging-window master CSV not found: {path}")

    df = pd.read_csv(path)

    required = ["station", "mid", "x_mid", "y_mid"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(
            f"Kriging-window master CSV missing required columns {missing}: {path}"
        )

    df = df.copy()
    df["station"] = df["station"].map(norm_station)
    df["mid"] = pd.to_datetime(df["mid"], errors="coerce").dt.normalize()

    if "w0" in df.columns:
        df["w0"] = pd.to_datetime(df["w0"], errors="coerce")
    else:
        df["w0"] = pd.NaT

    if "w1" in df.columns:
        df["w1"] = pd.to_datetime(df["w1"], errors="coerce")
    else:
        df["w1"] = pd.NaT

    df["x_mid"] = pd.to_numeric(df["x_mid"], errors="coerce")
    df["y_mid"] = pd.to_numeric(df["y_mid"], errors="coerce")

    valid = (
        df["station"].astype(str).ne("")
        & df["mid"].notna()
        & np.isfinite(df["x_mid"])
        & np.isfinite(df["y_mid"])
    )

    df = (
        df.loc[valid]
        .sort_values(["station", "mid"])
        .drop_duplicates(["station", "mid"], keep="last")
        .reset_index(drop=True)
    )

    if df.empty:
        raise RuntimeError(
            f"No valid station/mid/x_mid/y_mid rows remain in {path}"
        )

    return df

def windows_map_by_station(df_windows: pd.DataFrame) -> Dict[str, pd.DataFrame]:
    out: Dict[str, pd.DataFrame] = {}
    for st, g in df_windows.groupby("station", sort=False):
        out[st] = g.sort_values("mid").reset_index(drop=True)
    return out

def load_variogram_params_by_station(paths: List[Path]) -> Dict[str, Dict[str, Any]]:
    rows: List[pd.DataFrame] = []

    for path in paths:
        if not path.exists():
            raise FileNotFoundError(f"Variogram output CSV not found: {path}")
        d = pd.read_csv(path)
        required = ["station", RADIUS_COL]
        miss = [c for c in required if c not in d.columns]
        if miss:
            raise ValueError(f"Variogram output CSV missing columns {miss}: {path}")
        d = d.copy()
        d["source_variogram_csv"] = str(path)
        rows.append(d)

    if not rows:
        raise RuntimeError("No variogram output CSVs were configured.")

    df = pd.concat(rows, ignore_index=True)
    df["station"] = df["station"].map(norm_station)
    for c in ["x0_m", "y0_m", RADIUS_COL, "lat_mean", "lon_mean", "aRange_m"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    ok = (
        df["station"].astype(str).ne("")
        & np.isfinite(df[RADIUS_COL])
        & (df[RADIUS_COL] > 0)
    )
    df = df.loc[ok].copy()
    if df.empty:
        raise RuntimeError("No valid station rows remain after filtering variogram outputs.")

    # Later CSVs override earlier CSVs for duplicate stations.
    df = df.drop_duplicates(["station"], keep="last")
    return df.set_index("station").to_dict(orient="index")

class LoadStats(NamedTuple):
    n_loaded: int = 0
    n_time: int = 0
    n_bbox: int = 0
    n_circle: int = 0

def months_between(w0: pd.Timestamp, w1: pd.Timestamp) -> List[str]:
    start = pd.Timestamp(w0.year, w0.month, 1)
    end   = pd.Timestamp(w1.year, w1.month, 1)
    months = pd.date_range(start=start, end=end, freq="MS")
    if len(months) == 0:
        months = pd.date_range(start=start, end=start, freq="MS")
    w1m = pd.Timestamp(w1.year, w1.month, 1)
    if w1m > months[-1]:
        months = pd.date_range(start=start, end=w1m, freq="MS")
    return [m.strftime("%Y%m") for m in months]

def _cache_get(name: str) -> dict:
    d = getattr(_local_cache, name, None)
    if d is None:
        d = {}
        setattr(_local_cache, name, d)
    return d

def find_miswis_month_files(yyyymm: str) -> List[Path]:
    """
    Find MISWIS parquet files for one YYYYMM month across all configured
    directories.

    Search order:
      1) exact expected name: cs2_SINL2_YYYYMM.parquet
      2) fallback glob: *YYYYMM*.parquet

    Returned paths are unique and sorted by configured-directory order.
    """
    found: List[Path] = []
    seen: set[str] = set()

    for directory in MISWIS_MONTHLY_DIRS:
        exact = directory / f"{MISWIS_PREFIX}_{yyyymm}.parquet"

        candidates: List[Path] = []
        if exact.exists():
            candidates.append(exact)
        elif directory.exists():
            candidates.extend(sorted(directory.glob(f"*{yyyymm}*.parquet")))

        for path in candidates:
            key = str(path.resolve()) if path.exists() else str(path)
            if key not in seen:
                seen.add(key)
                found.append(path)

    return found

def _miswis_read_parquet_cached(path: Path) -> Optional[pd.DataFrame]:
    cache = _cache_get("miswis_parquet_cache")
    key = str(path)
    if key in cache:
        return cache[key]
    if not path.exists():
        return None
    df = pd.read_parquet(path, engine="pyarrow")
    cache[key] = df
    if len(cache) > _MISWIS_CACHE_MAX:
        cache.pop(next(iter(cache.keys())))
    return df

def load_miswis_window(
    w0: pd.Timestamp,
    w1: pd.Timestamp,
    x_mid: float,
    y_mid: float,
    R_req: float,
    xy_epsg: int,
) -> Tuple[pd.DataFrame, LoadStats]:
    frames: List[pd.DataFrame] = []
    loaded_total = 0

    for m in months_between(w0, w1):
        month_files = find_miswis_month_files(m)

        if not month_files:
            continue

        for p in month_files:
            d = _miswis_read_parquet_cached(p)
            if d is None or d.empty:
                continue

            dd = d.copy()
            dd["source_miswis_file"] = str(p)

            loaded_total += int(len(dd))
            frames.append(dd)

    if not frames:
        return pd.DataFrame(), LoadStats()

    df = pd.concat(frames, ignore_index=True)
    stats = LoadStats(n_loaded=loaded_total)

    if "time" not in df.columns:
        raise ValueError("MISWIS missing required column: time")
    df["time"] = parse_time_series(df["time"], origin=pd.Timestamp("2000-01-01"))

    need = ["time", "lon", "lat", "h_r1", "sigma0_r1"]
    missing = [c for c in need if c not in df.columns]
    if missing:
        raise ValueError(f"MISWIS parquet missing required columns: {missing}")

    # Preserve all measurement values. Only time/space define this subset.
    df = df.loc[(df["time"] >= w0) & (df["time"] < w1)].copy()
    stats = stats._replace(n_time=len(df))
    if df.empty:
        return df, stats

    lon = pd.to_numeric(df["lon"], errors="coerce").to_numpy(float)
    lat = pd.to_numeric(df["lat"], errors="coerce").to_numpy(float)
    ok_ll = np.isfinite(lon) & np.isfinite(lat)
    if not ok_ll.any():
        return df.iloc[0:0].copy(), stats

    x = np.full(lon.shape, np.nan, dtype=float)
    y = np.full(lat.shape, np.nan, dtype=float)
    if int(xy_epsg) == 3413:
        xx, yy = _ll2xy_3413(lon[ok_ll], lat[ok_ll])
    else:
        xx, yy = _ll2xy_3031(lon[ok_ll], lat[ok_ll])
    x[ok_ll] = np.asarray(xx, float)
    y[ok_ll] = np.asarray(yy, float)
    df["x"] = x
    df["y"] = y

    xmin, xmax = float(x_mid - R_req), float(x_mid + R_req)
    ymin, ymax = float(y_mid - R_req), float(y_mid + R_req)

    x = pd.to_numeric(df["x"], errors="coerce")
    y = pd.to_numeric(df["y"], errors="coerce")
    df = df.loc[
        np.isfinite(x) & np.isfinite(y) &
        (x >= xmin) & (x <= xmax) &
        (y >= ymin) & (y <= ymax)
    ].copy()
    stats = stats._replace(n_bbox=len(df))
    if df.empty:
        return df, stats

    df, _, _ = circle_subset_with_cap(df, x_mid, y_mid, R_req, cap_points=10**12)
    stats = stats._replace(n_circle=len(df))
    return df, stats

def _poca_read_parquet_cached(path: Path) -> Optional[pd.DataFrame]:
    cache = _cache_get("poca_parquet_cache")
    key = str(path)
    if key in cache:
        return cache[key]
    if not path.exists():
        return None
    df = pd.read_parquet(path, engine="pyarrow")
    cache[key] = df
    if len(cache) > _POCA_CACHE_MAX:
        cache.pop(next(iter(cache.keys())))
    return df

def load_poca_window(
    w0: pd.Timestamp, w1: pd.Timestamp, regions: List[str],
    x_mid: float, y_mid: float, R_req: float, xy_epsg: int
) -> Tuple[pd.DataFrame, LoadStats]:
    frames = []
    for m in months_between(w0, w1):
        for reg in regions:
            p = POCA_MONTHLY_DIR / f"{POCA_PREFIX}_{reg}_{m}.parquet"
            d = _poca_read_parquet_cached(p)
            if d is None or d.empty:
                continue
            dd = d.copy()
            dd["region_tag"] = reg
            frames.append(dd)

    if not frames:
        return pd.DataFrame(), LoadStats()

    df = pd.concat(frames, ignore_index=True)
    stats = LoadStats(n_loaded=len(df))

    need = ["time", "latitude", "longitude", "elevation", "uncertainty"]
    missing = [c for c in need if c not in df.columns]
    if missing:
        raise ValueError(f"POCA parquet missing required columns: {missing}")

    df["time"] = parse_time_series(df["time"], origin=pd.Timestamp("2000-01-01"))

    # Preserve all measurement values. Only time/space define this subset.
    df = df.loc[(df["time"] >= w0) & (df["time"] < w1)].copy()
    stats = stats._replace(n_time=len(df))
    if df.empty:
        return df, stats

    # project lon/lat -> x/y
    lon = pd.to_numeric(df["longitude"], errors="coerce").to_numpy(float)
    lat = pd.to_numeric(df["latitude"], errors="coerce").to_numpy(float)
    ok = np.isfinite(lon) & np.isfinite(lat)
    if not ok.any():
        return df.iloc[0:0].copy(), stats

    x = np.full(lon.shape, np.nan, dtype=float)
    y = np.full(lat.shape, np.nan, dtype=float)
    if int(xy_epsg) == 3413:
        xx, yy = _ll2xy_3413(lon[ok], lat[ok])
    else:
        xx, yy = _ll2xy_3031(lon[ok], lat[ok])
    x[ok] = np.asarray(xx, float)
    y[ok] = np.asarray(yy, float)
    df["x"] = x
    df["y"] = y

    # circle subset
    df, _, _ = circle_subset_with_cap(df, x_mid, y_mid, R_req, cap_points=10**12)
    stats = stats._replace(n_circle=len(df))
    return df, stats

def _swath_open_cached(path: Path):
    cache = _cache_get("swath_vaex_cache")
    key = str(path)
    if key in cache:
        return cache[key]
    if not path.exists():
        return None
    vdf = vaex.open(key)
    cache[key] = vdf
    if len(cache) > _SWATH_CACHE_MAX:
        cache.pop(next(iter(cache.keys())))
    return vdf

def load_swath_window_vaex(
    w0: pd.Timestamp, w1: pd.Timestamp, x_mid: float, y_mid: float, R: float, regions: List[str]
) -> Tuple[pd.DataFrame, LoadStats]:
    need = ["time", "x", "y", "elevation", "uncertainty"]
    out_frames: List[pd.DataFrame] = []
    stats = LoadStats()

    xmin, xmax = float(x_mid - R), float(x_mid + R)
    ymin, ymax = float(y_mid - R), float(y_mid + R)

    w0_s = float((pd.to_datetime(w0) - SWATH_TIME_ORIGIN) / pd.Timedelta(seconds=1))
    w1_s = float((pd.to_datetime(w1) - SWATH_TIME_ORIGIN) / pd.Timedelta(seconds=1))

    months = months_between(w0, w1)

    loaded_total = 0
    bbox_total = 0

    for reg in regions:
        for m in months:
            p = SWATH_MONTHLY_DIR / f"{SWATH_PREFIX}_{reg}_{m}.parquet"
            vdf = _swath_open_cached(p)
            if vdf is None:
                continue

            missing = [c for c in need if c not in vdf.get_column_names()]
            if missing:
                raise ValueError(f"SWATH parquet missing required columns: {missing}")

            tmask = (vdf["time"] >= w0_s) & (vdf["time"] < w1_s)
            bbox  = (vdf["x"] >= xmin) & (vdf["x"] <= xmax) & (vdf["y"] >= ymin) & (vdf["y"] <= ymax)

            vsub = vdf[tmask & bbox][need]
            pdf = vsub.to_pandas_df()
            loaded_total += int(len(pdf))
            if pdf.empty:
                continue

            bbox_total += int(len(pdf))
            pdf = pdf.copy()
            pdf["region_tag"] = reg
            pdf["time_raw"] = pd.to_numeric(pdf["time"], errors="coerce")
            pdf["time"] = parse_time_series(pdf["time_raw"], origin=SWATH_TIME_ORIGIN)
            # Preserve elevation/uncertainty here for the spatiotemporal-only export.
            pdf = pdf.dropna(subset=["time", "x", "y"]).copy()
            if not pdf.empty:
                out_frames.append(pdf)

    stats = stats._replace(n_loaded=loaded_total, n_bbox=bbox_total)
    if not out_frames:
        return pd.DataFrame(), stats
    df = pd.concat(out_frames, ignore_index=True)
    return df, stats

# ----------------------------------------------------------------------
# Select and export a single station/time window
# ----------------------------------------------------------------------

def _process_one_window(
    *,
    stU: str,
    w0: pd.Timestamp,
    w1: pd.Timestamp,
    mid: pd.Timestamp,
    R_req: float,
    x_mid: float,
    y_mid: float,
    lat_w: float,
    lon_w: float,
    center_source: str,
    center_method: str,
    center_day_used: pd.Timestamp,
    center_dt_days: float,
    xy_epsg: int,
    poca_regions: List[str],
    swath_regions: List[str],
    run_miswis_for_station: bool,
) -> Dict[str, Any]:
    if mid < POST_DATE_CUTOFF:
        return {"skipped": True, "subset_exported_counts": {}}

    mid_day = pd.to_datetime(mid).normalize()
    win_id_base = f"{stU}_{mid.strftime('%Y%m%dT%H%M%S')}_Rreq{int(R_req)}m"
    st_out_root = safe_makedirs(OUT_ROOT / stU)
    counts = {"MISWIS": 0, "POCA": 0, "SWATH": 0}

    # These fields are written in the same header format as the original.
    def base_attrs(ds: str, region: str) -> Dict[str, Any]:
        return dict(
            station=stU,
            dataset=ds,
            region=region,
            w0=w0,
            w1=w1,
            mid=mid,
            mid_day=mid_day,
            x_mid=x_mid,
            y_mid=y_mid,
            center_source=center_source,
            center_method=center_method,
            center_day_used=center_day_used,
            center_dt_days=center_dt_days,
            radius_requested_m=R_req,
            radius_used_m=R_req,
            xy_epsg=xy_epsg,
            poca_regions=",".join(poca_regions),
            swath_regions=",".join(swath_regions),
            gnss_bracketing_used="NO",
        )

    def needs_export(ds: str, win_id: str) -> bool:
        if not EXPORT_SPATIOTEMPORAL_SUBSETS:
            return False
        out_dir = st_out_root / ds
        out_csv = spatiotemporal_out_csv(out_dir, win_id)
        return not out_csv.exists()

    def export(ds: str, win_id: str, df: pd.DataFrame, region: str) -> None:
        out_dir = safe_makedirs(st_out_root / ds)
        ok, _ = export_spatiotemporal_subset(
            ds=ds, stU=stU, win_id=win_id,
            out_dir=out_dir, df=df, attrs=base_attrs(ds, region),
        )
        if ok:
            counts[ds] += 1

    # Standard CryoSat-2 L2 observations (MISWIS)
    ds = "MISWIS"
    win_id = f"{win_id_base}_MISWIS"
    if DO_MISWIS and run_miswis_for_station and needs_export(ds, win_id):
        df_raw, stats = load_miswis_window(
            w0, w1, x_mid=x_mid, y_mid=y_mid, R_req=R_req, xy_epsg=xy_epsg
        )
        export(ds, win_id, df_raw, "miswis")

    # CryoTEMPO Land Ice POCA
    ds = "POCA"
    win_id = f"{win_id_base}_POCA_MIXED"
    if DO_POCA and needs_export(ds, win_id):
        df_raw, stats = load_poca_window(
            w0, w1, regions=poca_regions, x_mid=x_mid,
            y_mid=y_mid, R_req=R_req, xy_epsg=xy_epsg,
        )
        region = (
            ",".join(sorted(df_raw["region_tag"].astype(str).unique().tolist()))
            if (not df_raw.empty and "region_tag" in df_raw.columns)
            else "UNKNOWN"
        )
        export(ds, win_id, df_raw, region)

    # CryoTEMPO swath: after time/bounding box, retain the same circular cut.
    ds = "SWATH"
    win_id = f"{win_id_base}_SWATH_MIXED"
    if DO_SWATH and needs_export(ds, win_id):
        df0, stats = load_swath_window_vaex(
            w0, w1, x_mid, y_mid, R_req, regions=swath_regions
        )
        df_raw = df0.copy()
        if not df_raw.empty:
            df_raw, _, _ = circle_subset_with_cap(
                df_raw, x_mid, y_mid, R_req, cap_points=10**12
            )
        region = (
            ",".join(sorted(df_raw["region_tag"].astype(str).unique().tolist()))
            if (not df_raw.empty and "region_tag" in df_raw.columns)
            else "UNKNOWN"
        )
        export(ds, win_id, df_raw, region)

    return {"skipped": False, "subset_exported_counts": counts}

def process_station(
    station: str,
    params_by_station: Dict[str, Dict[str, Any]],
    icefall_windows_by_station: Dict[str, pd.DataFrame],
):
    stU = norm_station(station).upper()
    _p(f"\n==============================\nSTATION {stU}\n==============================")

    params = params_by_station.get(stU)
    if params is None:
        expected = ICEFALL_VARIOGRAM_CSV
        _p(
            f"  [ERROR] Missing variogram parameter row for station {stU}. "
            f"Expected source: {expected}"
        )
        return

    try:
        R_req = float(params.get(RADIUS_COL, float("nan")))
    except Exception:
        R_req = float("nan")

    if not np.isfinite(R_req) or R_req <= 0:
        _p(
            f"  [ERROR] Missing/invalid radius for {stU}: "
            f"{RADIUS_COL}={params.get(RADIUS_COL)}"
        )
        return

    # Original Icefall file is REMA / EPSG:3031. The newer file carries
    # dem_name when available, which lets this also support ArcticDEM rows.
    dem_name = str(params.get("dem_name", "")).strip()
    if not dem_name:
        dem_name = "REMA"

    xy_epsg = xy_epsg_for_dem_name(dem_name)
    poca_regions, swath_regions = regions_for_dem_name(dem_name)
    run_miswis_for_station = True

    # Diagnostic station-level reference center only. It is not used as the
    # moving per-window center.
    try:
        x_mid_dem = float(params.get("x0_m", float("nan")))
        y_mid_dem = float(params.get("y0_m", float("nan")))
    except Exception:
        x_mid_dem = y_mid_dem = float("nan")

    half = pd.Timedelta(days=TIME_WINDOW_DAYS // 2)
    wins: List[Dict[str, Any]] = []

    # GZ / LA / MG: use the original Icefall kriging-window master directly.
    rows = icefall_windows_by_station.get(stU)
    if rows is None or rows.empty:
        _p(
            "  [SKIP] no valid kriging-window master rows found for this station "
            f"in {ICEFALL_KRIGING_WINDOWS_MASTER_CSV}"
        )
        return

    rows = (
        rows.dropna(subset=["mid", "x_mid", "y_mid"])
        .sort_values("mid")
        .drop_duplicates(["mid"], keep="last")
        .reset_index(drop=True)
    )

    for _, row in rows.iterrows():
        mid = pd.to_datetime(row["mid"], errors="coerce")
        x_mid = pd.to_numeric(pd.Series([row["x_mid"]]), errors="coerce").iloc[0]
        y_mid = pd.to_numeric(pd.Series([row["y_mid"]]), errors="coerce").iloc[0]

        if pd.isna(mid) or not (np.isfinite(x_mid) and np.isfinite(y_mid)):
            continue

        mid = mid.normalize()
        x_mid = float(x_mid)
        y_mid = float(y_mid)

        w0 = pd.to_datetime(row.get("w0", pd.NaT), errors="coerce")
        w1 = pd.to_datetime(row.get("w1", pd.NaT), errors="coerce")
        if pd.isna(w0):
            w0 = mid - half
        if pd.isna(w1):
            w1 = mid + half

        lon_w, lat_w = project_xy_to_lonlat(x_mid, y_mid, xy_epsg)

        wins.append(
            dict(
                w0=w0,
                w1=w1,
                mid=mid,
                lat_w=float(lat_w),
                lon_w=float(lon_w),
                x_mid=x_mid,
                y_mid=y_mid,
                center_source=f"{ICEFALL_KRIGING_WINDOWS_MASTER_CSV}::x_mid,y_mid",
                center_method="kriging_windows_master_x_mid_y_mid",
                center_day_used=mid,
                day_diff=0.0,
            )
        )

    _p(
        f"  window/center source: {ICEFALL_KRIGING_WINDOWS_MASTER_CSV} "
        "(mid, w0/w1 when present, x_mid, y_mid)"
    )

    variogram_source = params.get("source_variogram_csv", "UNKNOWN")
    _p(f"  variogram/radius source: {variogram_source}")

    if np.isfinite(x_mid_dem) and np.isfinite(y_mid_dem):
        _p(
            f"  DEM/source row: dem_name={dem_name} "
            f"reference x0_m={x_mid_dem:.3f}, y0_m={y_mid_dem:.3f}"
        )
    else:
        _p(
            f"  DEM/source row: dem_name={dem_name} "
            "(x0_m/y0_m unavailable; not needed for window centers)"
        )

    _p(f"  spatial projection: EPSG:{xy_epsg}")
    _p(f"  dataset regions: POCA={poca_regions}, SWATH={swath_regions}")
    _p(f"  radius request: R_req={R_req:.1f} m ({RADIUS_COL})")
    _p(f"  candidate windows with valid centers: {len(wins)}")

    if not wins:
        return

    first = wins[0]
    last = wins[-1]
    _p(
        f"  first center: mid={first['mid'].date()} "
        f"x={first['x_mid']:.3f} y={first['y_mid']:.3f} "
        f"lat={first['lat_w']:.8f} lon={first['lon_w']:.8f}"
    )
    _p(
        f"  last center:  mid={last['mid'].date()} "
        f"x={last['x_mid']:.3f} y={last['y_mid']:.3f} "
        f"lat={last['lat_w']:.8f} lon={last['lon_w']:.8f}"
    )

    win_total = len(wins)
    max_w = int(max(1, min(WINDOW_MAX_WORKERS, win_total)))

    with cf.ThreadPoolExecutor(max_workers=max_w) as ex:
        futures = []
        for win in wins:
            futures.append(ex.submit(
                _process_one_window,
                stU=stU,
                w0=win["w0"], w1=win["w1"], mid=win["mid"],
                R_req=R_req, x_mid=win["x_mid"], y_mid=win["y_mid"],
                lat_w=win["lat_w"], lon_w=win["lon_w"],
                center_source=win["center_source"],
                center_method=win["center_method"],
                center_day_used=win["center_day_used"],
                center_dt_days=win["day_diff"], xy_epsg=xy_epsg,
                poca_regions=poca_regions, swath_regions=swath_regions,
                run_miswis_for_station=run_miswis_for_station,
            ))
        for fut in cf.as_completed(futures):
            try:
                fut.result()
            except Exception as error:
                _p(f"[ERROR] {stU}: {error}")

def run_all(stations: Optional[Union[List[str], str]] = None):
    safe_makedirs(OUT_ROOT)

    params_by_station = load_variogram_params_by_station([ICEFALL_VARIOGRAM_CSV])

    icefall_windows_df = load_kriging_windows_master(
        ICEFALL_KRIGING_WINDOWS_MASTER_CSV
    )
    icefall_windows_by_st = windows_map_by_station(icefall_windows_df)

    if stations is None:
        station_list = sorted(icefall_windows_by_st.keys())
    elif isinstance(stations, str):
        station_list = [norm_station(stations)]
    else:
        station_list = [norm_station(st) for st in stations]

    station_list = sorted(set(station_list))

    _p(f"[INFO] Exporting raw spatial/temporal subsets for {len(station_list)} stations")
    with cf.ThreadPoolExecutor(
        max_workers=min(PARALLEL_MAX_WORKERS, max(1, len(station_list)))
    ) as ex:
        futures = [
            ex.submit(
                process_station,
                st,
                params_by_station,
                icefall_windows_by_st,
            )
            for st in station_list
        ]
        for fut in cf.as_completed(futures):
            try:
                fut.result()
            except Exception as error:
                _p(f"[ERROR] station task failed: {error}")
    _p("[INFO] Done exporting SPATIOTEMPORAL_ONLY subsets")


# The 29 GNSS stations analyzed in the paper.
STATIONS = [
    "GZ01", "GZ02", "GZ06", "GZ09", "GZ13", "GZ17", "GZ18",
    "LA01", "LA02", "LA03", "LA04", "LA05", "LA06", "LA07", "LA08",
    "LA09", "LA10", "LA11", "LA12", "LA13", "LA14", "LA15", "LA16",
    "LA17", "LA18",
    "MG04", "MG05", "MG06", "MG07",
]

if __name__ == "__main__":
    run_all(stations=STATIONS)


# Apply the per-window standard-deviation filter

This section applies the paper’s standard-deviation filter to the raw subsets before block-median processing.


In [ ]:
from __future__ import annotations

"""Apply the per-window sigma clipping to previously exported raw subsets.

The clipping, deduplication, and CSV export functions below come from the
October 3, 2026 window-processing script. This standalone version reads the
existing SPATIOTEMPORAL_ONLY files instead of reopening CryoSat-2 parquet files.
It writes the same station/dataset/STD_FILTERED filenames and headers.

Running this script does not alter the raw CSV files. Existing filtered outputs
are preserved, except for the original SWATH 10,000-point redo rule.
"""

from pathlib import Path
from typing import Dict, Any, List, Optional, Tuple
import numpy as np
import pandas as pd

# The original pipeline's output tree.
# Set OUT_ROOT to the folder containing the station/dataset subset tree.
OUT_ROOT = Path("/Volumes/cornice/marquardt/data/interpWindow")
RAW_SUBDIR = "SPATIOTEMPORAL_ONLY"
FILTERED_SUBDIR = "STD_FILTERED"
CLIP_Z = 3.0
MAX_SD_CHANGE_PCT = 5.0
MIN_POINTS_PER_WINDOW = 4
CAP_POINTS_SWATH = 10**30
CAPPED_REDO_NPOINTS = 10000
HEADER_NPOINTS_LINE_1BASED = 13

# The 29 GNSS stations analyzed in the paper.
STATIONS = [
    "GZ01", "GZ02", "GZ06", "GZ09", "GZ13", "GZ17", "GZ18",
    "LA01", "LA02", "LA03", "LA04", "LA05", "LA06", "LA07", "LA08",
    "LA09", "LA10", "LA11", "LA12", "LA13", "LA14", "LA15", "LA16",
    "LA17", "LA18",
    "MG04", "MG05", "MG06", "MG07",
]
DATASETS = ["MISWIS", "POCA", "SWATH"]


def _p(msg: str) -> None:
    print(msg, flush=True)

def safe_makedirs(p: Path) -> Path:
    p.mkdir(parents=True, exist_ok=True)
    return p

def iterative_sigma_clip(
    df: pd.DataFrame,
    col: str,
    z: float,
    max_sd_change_pct: float,
) -> pd.DataFrame:
    """
    Iterative z-sigma clipping with the stopping behavior used for this study.

    Pass 1 is always committed if it removes points. Starting with pass 2,
    the candidate clipped set is evaluated before it is committed. If the
    population-SD change produced by that candidate is below
    ``max_sd_change_pct``, clipping stops and that candidate is NOT committed.

    The standard-deviation threshold applies to the change in standard deviation between
    clipping passes; it is not a limit on the percentage of points removed.
    """
    if df is None or df.empty or col not in df.columns:
        return df.iloc[0:0].copy() if df is not None else pd.DataFrame()

    cur = df.copy()
    cur[col] = pd.to_numeric(cur[col], errors="coerce")
    cur = cur[np.isfinite(cur[col].to_numpy(float))].copy()
    if cur.empty:
        return cur

    pass_number = 1

    while True:
        x = cur[col].to_numpy(float)
        mu = float(np.mean(x))
        sd_before = float(np.std(x, ddof=0))

        if (not np.isfinite(mu)) or (not np.isfinite(sd_before)) or sd_before == 0.0:
            break

        lo = mu - z * sd_before
        hi = mu + z * sd_before
        keep = (x >= lo) & (x <= hi)
        candidate = cur.loc[keep].copy()

        # Nothing to remove.
        if len(candidate) == len(cur):
            break

        # First pass is always committed if it removes points.
        if pass_number == 1:
            cur = candidate
            if cur.empty:
                break
            pass_number += 1
            continue

        # From pass 2 onward, test the candidate BEFORE committing it.
        if candidate.empty:
            break

        sd_after = float(np.std(candidate[col].to_numpy(float), ddof=0))
        if not np.isfinite(sd_after):
            break

        sd_change_pct = 100.0 * abs(sd_after - sd_before) / abs(sd_before)

        if sd_change_pct < float(max_sd_change_pct):
            break

        cur = candidate
        pass_number += 1

    return cur

def circle_subset_with_cap(df: pd.DataFrame, x_mid: float, y_mid: float, radius_m: float, cap_points: int):
    if df is None or df.empty:
        return df.iloc[0:0].copy(), float("nan"), False
    if "x" not in df.columns or "y" not in df.columns:
        return df.iloc[0:0].copy(), float("nan"), False

    x = pd.to_numeric(df["x"], errors="coerce").to_numpy(float)
    y = pd.to_numeric(df["y"], errors="coerce").to_numpy(float)
    ok = np.isfinite(x) & np.isfinite(y)
    if not ok.any():
        return df.iloc[0:0].copy(), float("nan"), False

    dx = x[ok] - float(x_mid)
    dy = y[ok] - float(y_mid)
    d2 = dx*dx + dy*dy

    r2 = float(radius_m) * float(radius_m)
    in_circle = d2 <= r2
    if not in_circle.any():
        return df.iloc[0:0].copy(), float("nan"), False

    ok_pos = np.flatnonzero(ok)
    in_pos = ok_pos[np.flatnonzero(in_circle)]
    sub = df.iloc[in_pos].copy().reset_index(drop=True)

    if len(sub) <= cap_points:
        return sub, float(radius_m), False

    dist = np.sqrt(d2[in_circle])
    k = int(cap_points)
    take = np.argpartition(dist, kth=k-1)[:k]
    take = take[np.argsort(dist[take])]
    sub_cap = df.iloc[in_pos[take]].copy().reset_index(drop=True)
    radius_used = float(np.max(dist[take]))
    return sub_cap, radius_used, True

def export_to_csv(df: pd.DataFrame, out_csv: Path, attrs: Dict[str, Any]) -> None:
    if df is None or df.empty:
        return
    safe_makedirs(out_csv.parent)
    with open(out_csv, "w", encoding="utf-8") as f:
        f.write("# =================================================\n")
        f.write("# Station window export - CSV format\n")
        f.write("# =================================================\n")
        for k, v in sorted(attrs.items()):
            if isinstance(v, pd.Timestamp):
                v = v.isoformat()
            f.write(f"# {k}: {v}\n")
        f.write("# =================================================\n")
        df.to_csv(f, index=False)
    _p(f"  [WROTE] {out_csv} rows={len(df)}")

def _read_line_1based(path: Path, line_no_1based: int) -> Optional[str]:
    try:
        with open(path, "r", encoding="utf-8", errors="replace") as f:
            for i, line in enumerate(f, start=1):
                if i == line_no_1based:
                    return line.rstrip("\n")
    except Exception:
        return None
    return None

def swath_should_write(out_csv: Path) -> Tuple[bool, str]:
    if not out_csv.exists():
        return True, "missing"
    lineN = _read_line_1based(out_csv, HEADER_NPOINTS_LINE_1BASED)
    if lineN is None:
        return False, "exists_unknown_header"
    target = f"# n_points: {CAPPED_REDO_NPOINTS}"
    if lineN.strip() == target:
        return True, "redo_capped_10000"
    return False, "exists_full"

def simple_exists_should_write(out_csv: Path) -> Tuple[bool, str]:
    if out_csv.exists():
        return False, "exists"
    return True, "missing"

def _dedup_key_cols(ds: str, df: pd.DataFrame) -> List[str]:
    cols: List[str] = []
    if ds == "MISWIS":
        for c in ["time", "x", "y", "h_r1", "sigma0_r1"]:
            if c in df.columns: cols.append(c)
    elif ds == "POCA":
        for c in ["time", "x", "y", "elevation", "uncertainty", "region_tag"]:
            if c in df.columns: cols.append(c)
    elif ds == "SWATH":
        for c in ["time_raw", "x", "y", "elevation", "uncertainty", "region_tag"]:
            if c in df.columns: cols.append(c)
        if "time_raw" not in cols and "time" in df.columns:
            cols.insert(0, "time")
    return [c for c in cols if c in df.columns]

def dedup_df(ds: str, df: pd.DataFrame) -> Tuple[pd.DataFrame, int]:
    if df is None or df.empty:
        return df, 0
    keys = _dedup_key_cols(ds, df)
    if not keys:
        return df, 0
    n0 = len(df)
    out = df.drop_duplicates(subset=keys, keep="first").copy()
    return out, (n0 - len(out))

def _clip_per_window(
    ds: str,
    df: pd.DataFrame,
    max_sd_change_pct: float,
) -> pd.DataFrame:
    if df is None or df.empty:
        return df.iloc[0:0].copy()

    if ds == "MISWIS":
        need = ["time", "x", "y", "sigma0_r1", "h_r1"]
        df = df.dropna(subset=[c for c in need if c in df.columns]).copy()
        if df.empty:
            return df
        df = iterative_sigma_clip(
            df, "sigma0_r1", z=CLIP_Z,
            max_sd_change_pct=max_sd_change_pct,
        )
        if df.empty:
            return df
        return iterative_sigma_clip(
            df, "h_r1", z=CLIP_Z,
            max_sd_change_pct=max_sd_change_pct,
        )

    if ds in ("POCA", "SWATH"):
        need = ["time", "x", "y", "uncertainty", "elevation"]
        df = df.dropna(subset=[c for c in need if c in df.columns]).copy()
        if df.empty:
            return df
        df = iterative_sigma_clip(
            df, "uncertainty", z=CLIP_Z,
            max_sd_change_pct=max_sd_change_pct,
        )
        if df.empty:
            return df
        return iterative_sigma_clip(
            df, "elevation", z=CLIP_Z,
            max_sd_change_pct=max_sd_change_pct,
        )

    return df

def _export_one_dataset_window(
    *,
    ds: str,
    stU: str,
    win_id: str,
    out_dir: Path,
    df: pd.DataFrame,
    attrs: Dict[str, Any],
) -> Tuple[bool, str, int]:
    """Write one filtered window and return its status and row count."""
    out_csv = out_dir / f"{win_id}.csv"

    if ds == "SWATH":
        should_write, _ = swath_should_write(out_csv)
    else:
        should_write, _ = simple_exists_should_write(out_csv)

    if not should_write:
        return False, "skip_exists", 0
    if df is None or df.empty:
        return False, "skip_empty", 0
    if len(df) < MIN_POINTS_PER_WINDOW:
        return False, "skip_too_few", int(len(df))

    export_to_csv(df, out_csv, attrs)
    return True, "exported", int(len(df))

# The raw CSV has comment metadata lines preceding its normal header row.
# Retain the original window metadata for the filtered CSV header.
BASE_METADATA_KEYS = {
    "station", "dataset", "region", "w0", "w1", "mid", "mid_day",
    "x_mid", "y_mid", "center_source", "center_method", "center_day_used",
    "center_dt_days", "radius_requested_m", "radius_used_m", "xy_epsg",
    "poca_regions", "swath_regions", "gnss_bracketing_used",
    "lon_w", "lat_w",
}


def read_raw_subset(path: Path):
    attrs = {}
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if not line.startswith("#"):
                break
            key, sep, value = line[2:].strip().partition(": ")
            if sep and key in BASE_METADATA_KEYS:
                attrs[key] = value

    # Round-trip float parsing minimizes numeric changes when reloading a CSV.
    df = pd.read_csv(path, comment="#", low_memory=False,
                     float_precision="round_trip")
    if "time" in df.columns:
        df["time"] = pd.to_datetime(df["time"], errors="coerce")
    return df, attrs


def filter_one_file(raw_path: Path, station: str, dataset: str):
    suffix = "_SPATIOTEMPORAL_ONLY.csv"
    win_id = raw_path.name.removesuffix(suffix)
    out_dir = raw_path.parent.parent / FILTERED_SUBDIR
    out_csv = out_dir / f"{win_id}.csv"

    # Apply the original existing-file behavior before loading a large CSV.
    if dataset == "SWATH":
        should_write, why = swath_should_write(out_csv)
    else:
        should_write, why = simple_exists_should_write(out_csv)
    if not should_write:
        return "skip_exists", 0

    df_raw, base = read_raw_subset(raw_path)

    df_final = df_raw.copy()

    if not df_final.empty:
        df_final, _ = dedup_df(dataset, df_final)

        if dataset == "SWATH" and len(df_final) > CAP_POINTS_SWATH:
            df_final, _, _ = circle_subset_with_cap(
                df_final,
                float(base["x_mid"]), float(base["y_mid"]),
                float(base["radius_requested_m"]),
                cap_points=CAP_POINTS_SWATH,
            )

        df_final = _clip_per_window(dataset, df_final, MAX_SD_CHANGE_PCT)

    # The original program stores the regions *remaining after filtering*.
    if dataset in ("POCA", "SWATH"):
        base["region"] = (
            ",".join(sorted(df_final["region_tag"].astype(str).unique().tolist()))
            if (not df_final.empty and "region_tag" in df_final.columns)
            else "UNKNOWN"
        )
    else:
        base["region"] = "miswis"

    if dataset == "SWATH":
        base["cap_points"] = CAP_POINTS_SWATH

    attrs = dict(base)
    attrs.update(
        clip_mode="per_window_iter_sigma",
        clip_z=CLIP_Z,
        clip_max_sd_change_pct=float(MAX_SD_CHANGE_PCT),
        n_points=int(len(df_final)),
    )

    ok, status, n = _export_one_dataset_window(
        ds=dataset, stU=station, win_id=win_id,
        out_dir=out_dir, df=df_final, attrs=attrs,
    )
    return status, n


def main():
    summary = {}
    for station in STATIONS:
        for dataset in DATASETS:
            raw_dir = OUT_ROOT / station / dataset / RAW_SUBDIR
            if not raw_dir.exists():
                continue

            raw_files = sorted(raw_dir.glob("*_SPATIOTEMPORAL_ONLY.csv"))
            for raw_path in raw_files:
                status, n = filter_one_file(raw_path, station, dataset)
                summary[status] = summary.get(status, 0) + 1
                if status == "exported":
                    print(f"{station} {dataset}: {n} points retained")

    print("\nfiltering complete:", summary)


if __name__ == "__main__":
    main()


# Block-median processing of the filtered data


In [ ]:
# Apply a block median to the filtered population.
# Block width is 0.10 × aRange, with no size limits or clamping.
# Use the REMA variogram parameters for all 29 paper stations.
# Numeric and datetime columns use their median; text columns use the
# first non-null value. Keep the original count in `block_n`.

from pathlib import Path
import io
import re

import numpy as np
import pandas as pd

# ============================================================
# SETTINGS
# ============================================================

ROOT = Path(
    "/Volumes/cornice/marquardt/data/interpWindow"
)

# ------------------------------------------------------------
# aRange metadata
# ------------------------------------------------------------

# This is rema_variogram_params_station_midpoint.csv from the REMA R script.
ICEFALL_VARIOGRAM_CSV = Path(
    "/Volumes/icefall/marquardt/results/bias_results/"
    "rema_variogram_params_station_midpoint.csv"
)

# ------------------------------------------------------------
# Datasets
# ------------------------------------------------------------

DATASETS = (
    "MISWIS",
    "POCA",
    "SWATH",
)

# ------------------------------------------------------------
# BOTH filtered populations
# ------------------------------------------------------------

INPUT_SUBDIRS = ("STD_FILTERED",)

# ------------------------------------------------------------
# Stations
# ------------------------------------------------------------

TARGET_STATIONS = [
    "GZ01", "GZ02", "GZ06", "GZ09", "GZ13", "GZ17", "GZ18",
    "LA01", "LA02", "LA03", "LA04", "LA05", "LA06", "LA07", "LA08",
    "LA09", "LA10", "LA11", "LA12", "LA13", "LA14", "LA15", "LA16",
    "LA17", "LA18",
    "MG04", "MG05", "MG06", "MG07",
]

# ------------------------------------------------------------
# Block size
# ------------------------------------------------------------

BLOCK_FACTOR = 0.10

# IMPORTANT:
#
# block_m = 0.10 * aRange_m
#
# EXACTLY.
#
# No MIN_BLOCK_M
# No MAX_BLOCK_M
# No np.clip()

# ------------------------------------------------------------
# Output
# ------------------------------------------------------------

OUT_PARENT = "blockmedian_by_arange"
OUT_TAG = "0p10x"

# False = preserve existing block-median outputs
# True  = replace/regenerate them
OVERWRITE = False

# ============================================================
# HELPERS
# ============================================================

def norm_station(value):
    return re.sub(
        r"[^A-Z0-9]",
        "",
        str(value).strip().upper(),
    )

def first_non_null(series):

    valid = series.dropna()

    if len(valid):
        return valid.iloc[0]

    return np.nan

def is_datetime_name(name):

    name = str(name).strip().lower()

    return (
        name in {
            "time",
            "date",
            "datetime",
            "timestamp",
        }
        or name.endswith(
            (
                "_time",
                "_date",
                "_dt",
                "_datetime",
                "_timestamp",
            )
        )
        or name.startswith(
            (
                "time_",
                "date_",
                "datetime_",
                "timestamp_",
            )
        )
    )

# ============================================================
# READ COMMENTED CSV
# ============================================================

def read_header_csv(path):

    metadata = {}
    table_lines = []
    in_table = False

    with path.open(
        "r",
        encoding="utf-8",
        errors="replace",
    ) as file:

        for line in file:

            if not in_table and line.startswith("#"):

                text = line[1:].strip()

                if ":" in text:

                    key, value = text.split(
                        ":",
                        1,
                    )

                    key = key.strip()

                    if key:
                        metadata[key] = value.strip()

                continue

            if not in_table and not line.strip():
                continue

            in_table = True

            table_lines.append(
                line
            )

    if not table_lines:

        raise ValueError(
            f"No CSV table found in:\n{path}"
        )

    dataframe = pd.read_csv(
        io.StringIO(
            "".join(table_lines)
        ),
        low_memory=False,
    )

    return metadata, dataframe

# ============================================================
# WRITE COMMENTED CSV
# ============================================================

def write_header_csv(
    path,
    metadata,
    dataframe,
):

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        "." + path.name + ".tmp"
    )

    try:

        with temporary_path.open(
            "w",
            encoding="utf-8",
            newline="",
        ) as file:

            file.write(
                "# =================================================\n"
            )

            file.write(
                "# Block-median preprocessed STD-filtered window CSV\n"
            )

            file.write(
                "# =================================================\n"
            )

            for key, value in sorted(
                metadata.items()
            ):

                if isinstance(
                    value,
                    pd.Timestamp,
                ):
                    value = value.isoformat()

                file.write(
                    f"# {key}: {value}\n"
                )

            file.write(
                "# =================================================\n"
            )

            dataframe.to_csv(
                file,
                index=False,
            )

        temporary_path.replace(
            path
        )

    except Exception:

        temporary_path.unlink(
            missing_ok=True
        )

        raise

# ============================================================
# LOAD ONE aRange CSV
# ============================================================

def read_arange_csv(path):

    if not path.exists():

        raise FileNotFoundError(
            f"Variogram CSV not found:\n{path}"
        )

    dataframe = pd.read_csv(
        path
    )

    required = {
        "station",
        "aRange_m",
    }

    missing = required.difference(
        dataframe.columns
    )

    if missing:

        raise ValueError(
            f"Missing columns {sorted(missing)} "
            f"in:\n{path}"
        )

    dataframe = dataframe[
        [
            "station",
            "aRange_m",
        ]
    ].copy()

    dataframe["station"] = (
        dataframe["station"]
        .map(norm_station)
    )

    dataframe["aRange_m"] = (
        pd.to_numeric(
            dataframe["aRange_m"],
            errors="coerce",
        )
    )

    valid = (
        dataframe["station"].ne("")
        & np.isfinite(
            dataframe["aRange_m"]
        )
        & dataframe["aRange_m"].gt(0)
    )

    dataframe = dataframe.loc[
        valid
    ].copy()

    dataframe = (
        dataframe
        .drop_duplicates(
            "station",
            keep="last",
        )
    )

    return dict(
        zip(
            dataframe["station"],
            dataframe["aRange_m"].astype(float),
        )
    )

# ============================================================
# ROUTE STATIONS TO CORRECT aRange SOURCE
# ============================================================

def load_aranges():

    aranges = read_arange_csv(
        ICEFALL_VARIOGRAM_CSV
    )

    return {
        station: {
            "aRange_m": float(aranges[station]),
            "source_csv": str(ICEFALL_VARIOGRAM_CSV),
        }
        for station in TARGET_STATIONS
        if station in aranges
    }

# ============================================================
# PREPARE COLUMNS FOR MEDIAN
# ============================================================

def coerce_columns_for_median(
    dataframe
):

    output = dataframe.copy()

    for column in output.columns:

        series = output[column]

        original_non_null = int(
            series.notna().sum()
        )

        # ----------------------------------------------------
        # Datetime
        # ----------------------------------------------------

        if is_datetime_name(
            column
        ):

            parsed_datetime = (
                pd.to_datetime(
                    series,
                    errors="coerce",
                )
            )

            parsed_non_null = int(
                parsed_datetime
                .notna()
                .sum()
            )

            if (
                original_non_null == 0
                or parsed_non_null
                == original_non_null
            ):

                output[column] = (
                    parsed_datetime
                )

                continue

        # ----------------------------------------------------
        # Numeric strings
        # ----------------------------------------------------

        if (
            pd.api.types.is_object_dtype(
                series
            )
            or pd.api.types.is_string_dtype(
                series
            )
        ):

            parsed_numeric = (
                pd.to_numeric(
                    series,
                    errors="coerce",
                )
            )

            parsed_non_null = int(
                parsed_numeric
                .notna()
                .sum()
            )

            if (
                original_non_null > 0
                and parsed_non_null
                == original_non_null
            ):

                output[column] = (
                    parsed_numeric
                )

    return output

# ============================================================
# XY BLOCK MEDIAN
# ============================================================

def xy_block_median(
    source_dataframe,
    block_m,
):

    if "x" not in source_dataframe.columns:

        raise ValueError(
            "Input file is missing x"
        )

    if "y" not in source_dataframe.columns:

        raise ValueError(
            "Input file is missing y"
        )

    if (
        not np.isfinite(block_m)
        or block_m <= 0
    ):

        raise ValueError(
            f"Invalid block size: {block_m}"
        )

    original_columns = list(
        source_dataframe.columns
    )

    dataframe = (
        coerce_columns_for_median(
            source_dataframe
        )
    )

    dataframe["x"] = pd.to_numeric(
        dataframe["x"],
        errors="coerce",
    )

    dataframe["y"] = pd.to_numeric(
        dataframe["y"],
        errors="coerce",
    )

    valid_xy = (
        np.isfinite(
            dataframe["x"]
        )
        & np.isfinite(
            dataframe["y"]
        )
    )

    dataframe = dataframe.loc[
        valid_xy
    ].copy()

    if dataframe.empty:

        return pd.DataFrame(
            columns=(
                original_columns
                + ["block_n"]
            )
        )

    # --------------------------------------------------------
    # Grid origin is determined independently for EACH window
    # --------------------------------------------------------

    x_origin = float(
        dataframe["x"].min()
    )

    y_origin = float(
        dataframe["y"].min()
    )

    epsilon = 1.0e-12

    dataframe["_block_ix"] = np.floor(
        (
            dataframe["x"]
            - x_origin
        )
        / block_m
        + epsilon
    ).astype("int64")

    dataframe["_block_iy"] = np.floor(
        (
            dataframe["y"]
            - y_origin
        )
        / block_m
        + epsilon
    ).astype("int64")

    # --------------------------------------------------------
    # Aggregations
    # --------------------------------------------------------

    aggregations = {}

    for column in original_columns:

        if (
            pd.api.types.is_numeric_dtype(
                dataframe[column]
            )
            or pd.api.types.is_datetime64_any_dtype(
                dataframe[column]
            )
        ):

            aggregations[column] = (
                "median"
            )

        else:

            aggregations[column] = (
                first_non_null
            )

    groups = dataframe.groupby(
        [
            "_block_ix",
            "_block_iy",
        ],
        sort=False,
        observed=True,
        dropna=False,
    )

    result = groups.agg(
        aggregations
    )

    result["block_n"] = (
        groups
        .size()
        .astype("int64")
    )

    result = result.reset_index(
        drop=True
    )

    return result[
        original_columns
        + ["block_n"]
    ]

# ============================================================
# PROCESS ONE FILE
# ============================================================

def process_file(
    source_csv,
    station,
    dataset,
    input_subdir,
    arange_m,
    arange_source_csv,
):

    station = norm_station(
        station
    )

    dataset = str(
        dataset
    ).strip().upper()

    # --------------------------------------------------------
    # EXACT block size
    #
    # NO clamp
    # --------------------------------------------------------

    block_m = (
        BLOCK_FACTOR
        * float(arange_m)
    )

    if (
        not np.isfinite(block_m)
        or block_m <= 0
    ):

        raise ValueError(
            f"Invalid block size "
            f"from aRange_m={arange_m}"
        )

    # --------------------------------------------------------
    # Keep outputs underneath their OWN filter population
    # --------------------------------------------------------

    out_dir = (
        ROOT
        / station
        / dataset
        / input_subdir
        / OUT_PARENT
        / OUT_TAG
    )

    out_name = (
        f"{source_csv.stem}"
        f"__bmed_{OUT_TAG}.csv"
    )

    out_csv = (
        out_dir
        / out_name
    )

    if (
        out_csv.exists()
        and not OVERWRITE
    ):

        print(
            f"[SKIP EXISTS] {out_csv}"
        )

        return {
            "outcome": "skip_exists",
            "out_csv": str(
                out_csv
            ),
            "n_pre": np.nan,
            "n_post": np.nan,
        }

    # --------------------------------------------------------
    # Read
    # --------------------------------------------------------

    (
        source_metadata,
        source_dataframe,
    ) = read_header_csv(
        source_csv
    )

    if source_dataframe.empty:

        print(
            f"[SKIP EMPTY] {source_csv}"
        )

        return {
            "outcome": "skip_empty",
            "out_csv": str(
                out_csv
            ),
            "n_pre": 0,
            "n_post": 0,
        }

    # --------------------------------------------------------
    # Block median
    # --------------------------------------------------------

    block_dataframe = (
        xy_block_median(
            source_dataframe,
            block_m=block_m,
        )
    )

    # --------------------------------------------------------
    # Metadata
    # --------------------------------------------------------

    metadata = dict(
        source_metadata
    )

    metadata.update(
        {
            "station": station,

            "dataset": dataset,

            "input_filter_subdir": (
                input_subdir
            ),

            "source_csv": str(
                source_csv
            ),

            "aRange_m_station": float(
                arange_m
            ),

            "aRange_source_csv": str(
                arange_source_csv
            ),

            "block_factor": float(
                BLOCK_FACTOR
            ),

            "block_m": float(
                block_m
            ),

            "block_method": (
                "xy_block_median"
            ),

            "block_rule": (
                f"block_m = "
                f"{BLOCK_FACTOR} * aRange_m"
            ),

            "block_minimum_applied": "NO",

            "block_maximum_applied": "NO",

            "block_clamping_applied": "NO",

            "block_grid_origin": (
                "minimum valid x/y "
                "within each source window"
            ),

            "n_points_pre_block": int(
                len(source_dataframe)
            ),

            "n_points": int(
                len(block_dataframe)
            ),

            "block_dir": str(
                out_dir
            ),
        }
    )

    # --------------------------------------------------------
    # Write
    # --------------------------------------------------------

    write_header_csv(
        out_csv,
        metadata,
        block_dataframe,
    )

    print(
        f"[WROTE] {out_csv}\n"
        f"        filter = {input_subdir}\n"
        f"        aRange = {arange_m:.6f} m\n"
        f"        block  = {block_m:.6f} m\n"
        f"        rows   = "
        f"{len(source_dataframe):,} "
        f"-> {len(block_dataframe):,}"
    )

    return {
        "outcome": "wrote",
        "out_csv": str(
            out_csv
        ),
        "n_pre": int(
            len(source_dataframe)
        ),
        "n_post": int(
            len(block_dataframe)
        ),
    }

# ============================================================
# LOAD aRANGES
# ============================================================

arange_by_station = (
    load_aranges()
)

# ============================================================
# CHECK ROOT
# ============================================================

if not ROOT.exists():

    raise FileNotFoundError(
        f"Interpolation root does not exist:\n{ROOT}"
    )

# ============================================================
# MAP STATION DIRECTORIES
# ============================================================

station_directory_map = {
    norm_station(path.name): path
    for path in ROOT.iterdir()
    if path.is_dir()
}

# ============================================================
# STATION LIST
# ============================================================

if TARGET_STATIONS is None:

    stations = sorted(
        station_directory_map
    )

else:

    stations = [
        norm_station(station)
        for station in TARGET_STATIONS
    ]

    stations = list(
        dict.fromkeys(
            stations
        )
    )

# ============================================================
# RUN
# ============================================================

summary = []

print(
    "========================================"
)

print(
    "BLOCK MEDIAN: filtered data"
)

print(
    "========================================"
)

print(
    f"[INFO] Root: {ROOT}"
)

print(
    f"[INFO] Filters: {INPUT_SUBDIRS}"
)

print(
    f"[INFO] Datasets: {DATASETS}"
)

print(
    f"[INFO] Target stations: {stations}"
)

print(
    f"[INFO] Overwrite: {OVERWRITE}"
)

print(
    f"[INFO] Block rule: "
    f"block_m = {BLOCK_FACTOR} * aRange_m"
)

print(
    "[INFO] Minimum block size: NONE"
)

print(
    "[INFO] Maximum block size: NONE"
)

print(
    "[INFO] Block-size clamping: DISABLED"
)

# ============================================================
# STATIONS
# ============================================================

for station in stations:

    station_directory = (
        station_directory_map.get(
            station
        )
    )

    if station_directory is None:

        print(
            f"\n[SKIP STATION] "
            f"No folder found for {station}"
        )

        continue

    # --------------------------------------------------------
    # Station aRange
    # --------------------------------------------------------

    arange_record = (
        arange_by_station.get(
            station
        )
    )

    if arange_record is None:

        expected_source = ICEFALL_VARIOGRAM_CSV

        print(
            f"\n[SKIP STATION] "
            f"No valid aRange_m for {station}\n"
            f"               expected source: "
            f"{expected_source}"
        )

        continue

    arange_m = float(
        arange_record[
            "aRange_m"
        ]
    )

    arange_source_csv = (
        arange_record[
            "source_csv"
        ]
    )

    # Exact 10%
    block_m = (
        BLOCK_FACTOR
        * arange_m
    )

    print(
        "\n"
        "========================================\n"
        f"{station}\n"
        f"aRange_m = {arange_m:.6f} m\n"
        f"block_m  = {block_m:.6f} m\n"
        f"source   = {arange_source_csv}\n"
        "========================================"
    )

    # ========================================================
    # DATASETS
    # ========================================================

    for dataset in DATASETS:

        dataset_directory = (
            station_directory
            / dataset
        )

        # ====================================================
        # filtered data PRODUCT
        # ====================================================

        for input_subdir in INPUT_SUBDIRS:

            input_directory = (
                dataset_directory
                / input_subdir
            )

            # Only direct CSV children.
            # NOT recursive.
            #
            # Therefore:
            #
            # blockmedian_by_arange/
            #
            # cannot be reprocessed accidentally.

            if input_directory.exists():

                source_files = sorted(
                    input_directory.glob(
                        "*.csv"
                    )
                )

            else:

                source_files = []

            # Extra safety
            source_files = [
                path
                for path in source_files
                if "__bmed_"
                not in path.stem.lower()
            ]

            if not source_files:

                print(
                    f"[NO FILES] "
                    f"{station}/"
                    f"{dataset}/"
                    f"{input_subdir}"
                )

                continue

            print(
                f"[{dataset} | {input_subdir}] "
                f"{len(source_files)} source CSV(s)"
            )

            # ================================================
            # FILES
            # ================================================

            for source_csv in source_files:

                try:

                    result = process_file(
                        source_csv=source_csv,
                        station=station,
                        dataset=dataset,
                        input_subdir=(
                            input_subdir
                        ),
                        arange_m=arange_m,
                        arange_source_csv=(
                            arange_source_csv
                        ),
                    )

                except Exception as error:

                    result = {
                        "outcome": "error",
                        "out_csv": "",
                        "n_pre": np.nan,
                        "n_post": np.nan,
                    }

                    print(
                        f"[ERROR] {source_csv}\n"
                        f"        "
                        f"{type(error).__name__}: "
                        f"{error}"
                    )

                summary.append(
                    {
                        "station": station,

                        "dataset": dataset,

                        "filter": (
                            input_subdir
                        ),

                        "source_csv": str(
                            source_csv
                        ),

                        "aRange_m": (
                            arange_m
                        ),

                        "block_factor": (
                            BLOCK_FACTOR
                        ),

                        "block_m": (
                            block_m
                        ),

                        "aRange_source_csv": (
                            arange_source_csv
                        ),

                        "outcome": (
                            result[
                                "outcome"
                            ]
                        ),

                        "n_points_pre_block": (
                            result[
                                "n_pre"
                            ]
                        ),

                        "n_points_post_block": (
                            result[
                                "n_post"
                            ]
                        ),

                        "out_csv": (
                            result[
                                "out_csv"
                            ]
                        ),
                    }
                )

# ============================================================
# FINAL SUMMARY
# ============================================================

blockmedian_summary = (
    pd.DataFrame(
        summary
    )
)

print(
    "\n"
    "========================================\n"
    "FINAL SUMMARY\n"
    "========================================"
)

if blockmedian_summary.empty:

    print(
        "No source files were found or processed."
    )

else:

    print(
        blockmedian_summary.groupby(
            [
                "station",
                "dataset",
                "filter",
                "outcome",
            ],
            dropna=False,
        )
        .size()
        .rename(
            "n_files"
        )
        .to_string()
    )

# Display detailed summary in Jupyter
blockmedian_summary
